# Нелинейные модели (KNN, SVM)
и логистическая регрессия

*ML-3.1 Способен осуществлять обоснованный и осознанный выбор классических
алгоритмов МО с учителем и выполнять их параметризацию в зависимости от
поставленной задачи и имеющихся данных и с пониманием математической
природы данных, вычислительных ограничений и требований к
интерпретируемости результатов*

Сделайте копию этого файла (Файл - Сохранить копию на диске), переименуйте её, добавив в название вашу фамилию. Например, Иванова_Линейные_Нелинейные.ipynb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_digits, load_wine, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    mean_squared_error, r2_score
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

Будем использовать набор данных load_digits — 1797 изображений рукописных цифр размером 8x8 пикселей. Этот датасет хорошо подходит для задач классификации и наглядно демонстрирует разницу между линейными и нелинейными моделями.

In [ ]:
digits = load_digits()
X = digits.data  # 64 признака (8x8 пикселей)
y = digits.target  # 10 классов (цифры 0-9)

print(f"Размерность данных: {X.shape}")
print(f"Количество классов: {len(np.unique(y))}")

# Посмотрим на несколько цифр
fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for i in range(5):
    axes[i].imshow(digits.images[i], cmap='gray')
    axes[i].set_title(f"Цифра: {y[i]}")
    axes[i].axis('off')
plt.show()

**Практика.**

Измените код выше так, чтобы:
- Отобразить цифры с индексами с 10 по 14
- Вывести в консоль форму изображений (images.shape)

**Вопрос:** Почему каждый пиксель является отдельным признаком? Сколько всего признаков в датасете и как это связано с размером изображения?

In [ ]:
# Разместите здесь ваш код

**Ваш ответ на вопрос:**

# Подготовка данных: стандартизация

Масштабируем данные, используя StandardScaler, который корректирует данные так, чтобы среднее значение было равно 0, а стандартное отклонение — 1. Это шаг обязателен для алгоритмов, основанных на расстояниях (KNN, SVM) и для логистической регрессии с регуляризацией.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Разделение на обучающую и тестовую выборки (70% / 30%)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")

**Вопросы для размышления (ответьте письменно):**

1. Почему стандартизация важна для KNN, SVM и логистической регрессии? Что произойдет, если не выполнить стандартизацию?

2. Почему мы применяем `fit_transform` к обучающей выборке, но только `transform` к тестовой?

**Ваш ответ:**

# Часть 1. Логистическая регрессия (базовая линейная модель)

Логистическая регрессия — это **линейная модель классификации**. Она служит "базовым уровнем" (baseline), с которым сравнивают более сложные модели. Если сложная модель не превосходит логистическую регрессию, то её использование не оправдано.

**Почему логистическая регрессия — это "базовый уровень"?**
- Она проста и быстро обучается
- Легко интерпретируется (веса признаков показывают их важность)
- Дает "нижнюю планку" качества: если сложная модель не лучше, она не нужна

На изображениях рукописных цифр логистическая регрессия работает с каждым пикселем как с отдельным признаком, но **не учитывает пространственную структуру** изображения.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Обучаем логистическую регрессию
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Предсказание
y_pred_lr = lr.predict(X_test)

# Оценка качества
print("=== Логистическая регрессия ===")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_lr):.4f}")
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_lr))

**Вопросы для размышления:**

1. Почему логистическая регрессия считается "линейной" моделью? Что это значит для разделения классов?

2. Какая точность у логистической регрессии на digits? Как вы думаете, почему она такая?

3. Посмотрите на веса признаков (lr.coef_). Как вы думаете, почему некоторые пиксели имеют больший вес, чем другие?

In [ ]:
# Визуализация весов логистической регрессии
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    weight_image = lr.coef_[i].reshape(8, 8)
    ax.imshow(weight_image, cmap='RdBu', vmin=-np.abs(weight_image).max(), vmax=np.abs(weight_image).max())
    ax.set_title(f'Цифра {i}')
    ax.axis('off')
plt.suptitle('Веса признаков для каждой цифры (красный — важные пиксели)', fontsize=14)
plt.tight_layout()
plt.show()

print("\nВопрос: Какие пиксели оказались наиболее важными для распознавания каждой цифры?")
print("Почему некоторые пиксели имеют отрицательные веса?")

**Ваш ответ на вопросы:**

# Часть 2. Метод K-ближайших соседей (KNN)

KNN — один из простейших нелинейных алгоритмов классификации, основанный на принципе "скажи мне, кто твой друг, и я скажу, кто ты". Алгоритм классифицирует новый объект на основе класса большинства среди его k ближайших соседей в пространстве признаков.

**В отличие от логистической регрессии:**
- KNN не строит модель, а просто запоминает данные (ленивое обучение)
- KNN может выделять нелинейные границы между классами
- KNN чувствителен к масштабу признаков и выбросам

In [ ]:
# Создаем модель KNN с k=5 (стандартное значение)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Предсказание на тестовой выборке
y_pred_knn = knn.predict(X_test)

# Оценка качества
print("=== KNN (k=5) ===")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_knn):.4f}")
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_knn))

**Практика.**

Измените код выше: обучите модель KNN с `k=1` и с `k=15`.

**Вопросы для размышления:**
- Как изменилась точность при k=1? Почему это называется "переобучением"?
- Как изменилась точность при k=15? Почему это может быть "недообучением"?
- Какое из этих значений k (1, 5, 15) дало лучший результат? Почему?
- Сравните точность KNN (k=5) с логистической регрессией. Какая модель лучше и почему?

In [ ]:
# Разместите здесь ваш код для k=1 и k=15

**Ваш ответ на вопросы:**

**Практика.**

Используйте GridSearchCV для подбора оптимального значения k в диапазоне от 1 до 30. Визуализируйте зависимость качества от k.

**Вопрос:** Какое значение k оказалось оптимальным? Как вы думаете, почему именно это значение?

In [ ]:
# Разместите здесь ваш код для GridSearchCV

**Ваш ответ на вопрос:**

**Практика.**

Сравните две модели KNN с оптимальным k:
- С `weights='uniform'` (все соседи равноправны)
- С `weights='distance'` (близкие соседи имеют больший вес)

**Вопрос:** Какая модель показала лучшую точность? Почему? В каких задачах использование `weights='distance'` может быть нежелательным?

In [ ]:
# Разместите здесь ваш код для сравнения весов

**Ваш ответ на вопрос:**

# Часть 3. Метод опорных векторов (SVM)

SVM — мощный алгоритм классификации, который строит гиперплоскость (или множество гиперплоскостей) в многомерном пространстве, максимально разделяющую классы. Для нелинейных данных SVM использует "ядро" (kernel trick), чтобы отобразить данные в пространство более высокой размерности, где они становятся линейно разделимыми.

**В отличие от логистической регрессии:**
- SVM может строить нелинейные границы (с помощью ядер)
- SVM максимизирует отступ (margin) между классами
- SVM использует только опорные векторы, а не все данные

In [ ]:
# Создаем модель SVM с RBF-ядром (параметры по умолчанию)
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("=== SVM (RBF, C=1, gamma='scale') ===")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_svm))

**Практика.**

Измените код выше: обучите SVM с линейным ядром (`kernel='linear'`).

**Вопрос:** Как изменилась точность? Почему RBF-ядро часто работает лучше на данных типа изображений? Сравните с логистической регрессией — почему линейный SVM и логистическая регрессия показывают похожие результаты?

In [ ]:
# Разместите здесь ваш код для SVM с линейным ядром

**Ваш ответ на вопрос:**

**Практика.**

Используйте GridSearchCV для подбора оптимальных значений C и gamma для SVM с RBF-ядром. Визуализируйте результаты в виде тепловой карты.

**Вопросы:**
- Какие значения C и gamma оказались оптимальными?
- Что контролирует параметр C? Что произойдет при очень большом C?
- Что контролирует параметр gamma? Что произойдет при очень большой gamma?

In [ ]:
# Разместите здесь ваш код для GridSearchCV

**Ваш ответ на вопросы:**

# Часть 4. Сравнение всех моделей

Сравним все три модели: Логистическая регрессия (линейная), KNN (нелинейная, ленивая), SVM (нелинейная с ядром).

**Практика.**

Сравните все три модели:
1. Логистическая регрессия (базовый уровень)
2. KNN (оптимальное k)
3. SVM (оптимальные C и gamma)

Постройте матрицы ошибок для всех моделей и сравните точность и время предсказания.

**Вопросы для размышления:**
1. Какая модель показала лучшую точность? Почему?

2. Логистическая регрессия — самая быстрая или самая медленная? Почему?

3. Какая модель проще для интерпретации? Почему?

4. Где логистическая регрессия показывает результаты, близкие к KNN/SVM, а где — нет? Как вы думаете, почему?

5. Если бы вы решали задачу распознавания цифр в мобильном приложении, какую модель выбрали бы и почему?

6. Если бы вы решали задачу медицинской диагностики, какую модель выбрали бы и почему?

In [ ]:
# Разместите здесь ваш код для сравнения

**Ваш ответ на вопросы:**

# Часть 5. Дополнительное задание: Задача регрессии (по желанию)

Для закрепления понимания различий между линейными и нелинейными моделями, выполните аналогичное сравнение на задаче **регрессии**.

**Практика.**

Используйте датасет load_diabetes (задача регрессии, 10 признаков, 442 объекта).

1. Загрузите данные, разделите на обучающую и тестовую выборки, выполните стандартизацию.

2. Обучите **линейную регрессию** (LinearRegression).

3. Обучите **KNN-регрессор** (KNeighborsRegressor) с подбором оптимального k.

4. Сравните модели по метрикам: MSE (Mean Squared Error) и R².

**Вопросы для сравнения:**
- Какая модель показала лучшее качество на задаче регрессии?
- Почему на этом датасете линейная регрессия может работать хорошо, а на digits — нет?
- Как число признаков и размер выборки влияют на выбор между линейной и нелинейной моделью?

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Разместите здесь ваш код

**Ваш ответ на вопросы:**

# Часть 6. Дополнительное задание: Сравнение на Wine (по желанию)

Повторите сравнение всех трех моделей (логистическая регрессия, KNN, SVM) на датасете load_wine (13 признаков, 3 класса).

**Практика.**

Повторите все шаги для датасета load_wine.

**Вопросы для сравнения:**
- Как отличается точность моделей на Wine по сравнению с Digits?
- Почему на Wine логистическая регрессия показывает высокую точность, а на Digits — заметно ниже?
- Как число признаков влияет на выбор между линейной и нелинейной моделью?

In [ ]:
from sklearn.datasets import load_wine

# Разместите здесь ваш код

**Ваш ответ на вопросы:**

# Итоговые выводы

**Практика.**

Напишите краткое эссе (7-10 предложений), в котором:

1. Сравните все три модели по трем критериям: точность, скорость, интерпретируемость

2. Объясните, почему логистическая регрессия — это "базовый уровень" (baseline) и зачем его использовать

3. Опишите, в каких задачах вы бы использовали каждый из алгоритмов:
   - Логистическую регрессию
   - KNN
   - SVM

4. Объясните, как структура данных (количество признаков, размер выборки, линейность) влияет на выбор алгоритма

**Рекомендация:** Используйте результаты ваших экспериментов для аргументации.

**Ваше эссе:**